<a href="https://colab.research.google.com/github/JAVERIAADIL/Learning-GPU-infrastructure/blob/module1/TiledNaiveMul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cupy as cp
from numba import cuda
from numba import float32


In [ ]:
# vector_1 = [[4,5,6],[7,8,9],[1,2,3]]
# vector_2 = [[1,2,3],[4,5,6],[7,8,9]]
# result = [[0,0,0],[0,0,0],[0,0,0]]

In [ ]:
vector_1 = cp.random.randn(4096,4096)
vector_2 = cp.random.randn(4096,4096)
result = cp.zeros((4096,4096))

In [ ]:
# cupy_vector1 = cp.array(vector_1).flatten()
# cupy_vector2 = cp.array(vector_2).flatten()
# c_result = cp.array(result).flatten()

In [ ]:
cupy_vector1 = cp.array(vector_1)
cupy_vector2 = cp.array(vector_2)
c_result = cp.array(result)

In [ ]:
d_vector1 = cuda.to_device(cupy_vector1)
d_vector2 = cuda.to_device(cupy_vector2)
d_result = cuda.to_device(c_result)

In [ ]:
N = 4096 # as 2 (128,128) metrices
block = (32,32) #as 1024 thread which give output of multiplication of 2 metrices comes in shared memory as tile of (32, 32)
grid=(N//32, N//32)

Explanation:

In [ ]:
# @cuda.jit
# def tiledMul(d_vector1,d_vector2, d_result, N):
#   tileA = cuda.shared.array(shape=(32, 32), dtype=float32) #this is how we define shared array in cp/python
#   tileB = cuda.shared.array(shape=(32, 32), dtype=float32)
#   tx = cuda.threadIdx.x; #it give thread number on x axis like if there is 128 * 128 metrics and one thread have to fetch row 1 and column 2 value which is 3rd value in row so thread assign by gpu as (2,1) which is (tx,ty) so tx= 2 and ty = 1 as colno = tx and row no = ty
#   ty = cuda.threadIdx.y;
#   row = cuda.blockIdx.y * cuda.blockDim.y + ty # here one thread assign to one complete row of 128 values and same for column
#   column = cuda.blockIdx.x * cuda.blockDim.x + tx
# partial_sum = 0

#   for step in range(N/32): # here loading from HBM to shared memory
#     tileA[ty][tx] = d_vector1[row * N + (step * 32 + tx)]; # in our case step is 0 to 3 means 4 step as 128*128 comes in 4 steps of 32*32
#     tileB[ty][tx] = d_vector2[(step * 32 + ty) + N * column];
#     # so like if thread (2,1) means it have to pickup row 1 column 2 value  or row 33 col 34 if it is second batch of loading shared memory and place in tile[2][1] which is tile[ty][tx] so it goes into d_vector1 pick [row = 1 * N=128 as we are considering array as id so jump is 128 + (step = 1 if it is second batch * 32 jump by 32  times more as one batch is already computed + 2)]
#     cuda.__syncthreads()
#     # we are syncing it because we have to make sure one 32* 32 metrix come first it calculate its partial sum then load another batch otherwise calculation go wrong

#     for k in range(32):
#       partial_sum += tileA[ty][k] * tileB[k][tx] #now in 32*32 metrics we perform multiplation like ty is row number then k its that value in row and tile b is like k=2 means 2nd row tx = 1 column so its is 3rd value in 2nd column

#       cuda.__syncthreads()
#   d_result[row * N + column] = partial_sum; #now put that partail_sum into 128*128 metrics



Code for 1d:

In [ ]:
# @cuda.jit
# def tiledMul(d_vector1,d_vector2, d_result, N):
#   tileA = cuda.shared.array(shape=(32, 32), dtype=float32)
#   tileB = cuda.shared.array(shape=(32, 32), dtype=float32)
#   partial_sum = 0
#   tx = cuda.threadIdx.x;
#   ty = cuda.threadIdx.y;
#   row = cuda.blockIdx.y * cuda.blockDim.y + ty
#   column = cuda.blockIdx.x * cuda.blockDim.x + tx

#   for step in range(N/32):
#     tileA[ty][tx] = d_vector1[row * N + (step * 32 + tx)];
#     tileB[ty][tx] = d_vector2[(step * 32 + ty) + N * column];

#     cuda.__syncthreads()


#     for k in range(32):
#       partial_sum += tileA[ty][k] * tileB[k][tx]

#       cuda.__syncthreads()
#   d_result[row * N + column] = partial_sum;



Code for 2d:

In [ ]:
@cuda.jit
def tiledMul(d_vector1,d_vector2, d_result, N):
  tileA = cuda.shared.array(shape=(32, 32), dtype=float32)
  tileB = cuda.shared.array(shape=(32, 32), dtype=float32)
  partial_sum = float32(0.0)
  tx = cuda.threadIdx.x;
  ty = cuda.threadIdx.y;
  row = cuda.blockIdx.y * cuda.blockDim.y + ty # here one thread assign to one complete row of 128 values and same for column
  column = cuda.blockIdx.x * cuda.blockDim.x + tx

  for step in range(N//32): # here loading from HBM to shared memory
    tileA[ty][tx] = d_vector1[row][step * 32 + tx];
    tileB[ty][tx] = d_vector2[step * 32 + ty][column];

    cuda.syncthreads()

    for k in range(32):
      partial_sum += tileA[ty][k] * tileB[k][tx]

    cuda.syncthreads()
  d_result[row][column] = partial_sum;

In [ ]:
@cuda.jit
def vectorMul(d_vector1, d_vector2, d_result_arg):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    column = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if ( row < N and column < N):
      sum = 0
      for K in range(N):
        sum += d_vector1[row][K] * d_vector2[K] [column]
      d_result_arg[row][column] = sum

In [ ]:
# Warmup first — always
vectorMul[grid, block](d_vector1, d_vector2, d_result)
cuda.synchronize()

# Now measure
start_event = cuda.event(timing=True)
end_event   = cuda.event(timing=True)

cuda.synchronize()          # wait for anything pending
start_event.record()

vectorMul[grid, block](d_vector1, d_vector2, d_result)

end_event.record()
end_event.synchronize()     # wait for kernel to finish
naive_time = start_event.elapsed_time(end_event)

# Print time
print(f"GPU time for Naive Multiplication: {start_event.elapsed_time(end_event):.3f} ms")

# Copy result back and print
result = d_result.copy_to_host()
print("Result:", result)

GPU time for Naive Multiplication: 1109.728 ms
Result: [[-85.13710828 -58.32296674 -77.41089589 ... -48.00796638  43.70374663
  -19.57421395]
 [ 26.84061327 135.54641984  67.56458243 ... -32.0407424  -34.87996727
   -6.84837573]
 [-47.67272725 -76.82345119  79.5267124  ...  42.51356493 -26.4088516
   63.24923196]
 ...
 [ 25.44678012 -22.08863953  14.29956388 ...  38.35908533  33.55168029
   10.92028974]
 [-17.70595418 -33.98943002 -13.45794845 ...   3.98644416   4.26492836
   15.10052757]
 [ 21.56654572 -86.79921045 -98.18345104 ... -53.42068938 -61.39688457
    2.88812945]]


In [ ]:
# Warmup first — always
tiledMul[grid, block](d_vector1, d_vector2, d_result, N)
cuda.synchronize()

# Now measure
start_event = cuda.event(timing=True)
end_event   = cuda.event(timing=True)

cuda.synchronize()          # wait for anything pending
start_event.record()

tiledMul[grid, block](d_vector1, d_vector2, d_result, N)

end_event.record()
end_event.synchronize()     # wait for kernel to finish

tiled_time = start_event.elapsed_time(end_event)

# Print time
print(f"GPU time for Tiled Multiplication: {start_event.elapsed_time(end_event):.3f} ms")

# Copy result back and print
result = d_result.copy_to_host()
print("Result:", result)

GPU time for Tiled Multiplication: 213.811 ms
Result: [[-85.13718414 -58.32313538 -77.41100311 ... -48.00797653  43.70375443
  -19.57417297]
 [ 26.84058762 135.54621887  67.56458282 ... -32.04072571 -34.87995529
   -6.84839153]
 [-47.67268753 -76.82352448  79.5266037  ...  42.51357651 -26.40881729
   63.24922562]
 ...
 [ 25.44675446 -22.08864403  14.29962349 ...  38.35910034  33.55171585
   10.92027569]
 [-17.70591736 -33.98941422 -13.45795631 ...   3.98642802   4.2649641
   15.10056496]
 [ 21.56652832 -86.79911041 -98.18338776 ... -53.42058182 -61.39693069
    2.88817   ]]


In [ ]:
speed_up = tiled_time/naive_time *100
print(f"speed up:{speed_up}%")

speed up:19.266984699404503%


Observation:
For small metric size like 128, 128 naive multiplication time is 0.281ms while tiled multiplication time is 0.441 ms because this size of metric easily fit in L2 cache automatically and there is no need to use shared memory but if metric size is larger like 4096 then tiled(shared memory ) version is faster 213.881 ms and naive one is slower because now every thread go into HBRM to fetch data and speed is almost 19%